# Render Custom World
Visualize a custom world `.txt` file as it would appear in Crafter.

In [ ]:
# === Change this path to your custom world file ===
WORLD_PATH = '../custom_worlds/maze.txt'
TILE_PX = 16  # pixels per tile (increase for higher res)

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.path.dirname(os.path.abspath('.')), ''))

import numpy as np
import matplotlib.pyplot as plt
import crafter
import crafter.engine as crafter_engine
import crafter.objects as crafter_objects

# Import the material/object code mappings from our env wrapper
from embodied.envs.crafter import MATERIAL_CODES, OBJECT_CODES

In [ ]:
def parse_world_file(path):
    """Parse a custom world text file into a 2D grid."""
    path = os.path.expanduser(path)
    with open(path) as f:
        lines = [line.strip() for line in f if line.strip()]
    grid = [line.split() for line in lines]
    n_rows = len(grid)
    n_cols = len(grid[0])
    for r, row in enumerate(grid):
        assert len(row) == n_cols, f"Row {r} has {len(row)} cols, expected {n_cols}"
        for c, ch in enumerate(row):
            assert ch in MATERIAL_CODES or ch in OBJECT_CODES, \
                f"Unknown tile code '{ch}' at row {r}, col {c}"
    return grid, (n_cols, n_rows)


def render_world(grid, area, tile_px=16):
    """Render custom world by compositing textures tile-by-tile."""
    n_cols, n_rows = area
    # Create a throwaway env just to get the texture atlas
    env = crafter.Env(area=(4, 4), size=(64, 64), reward=False, seed=0)
    env.reset()
    textures = env._textures
    unit = (tile_px, tile_px)

    image = np.zeros((n_rows * tile_px, n_cols * tile_px, 3), dtype=np.uint8)
    for row_idx, row in enumerate(grid):
        for col_idx, ch in enumerate(row):
            # Get material texture (objects sit on grass)
            mat_name = MATERIAL_CODES.get(ch, 'grass') if ch not in OBJECT_CODES else 'grass'
            tex = textures.get(mat_name, unit)
            if tex.shape[-1] == 4:
                tex = tex[..., :3]
            # Crafter textures are in (x, y, c) order — transpose to (y, x, c)
            tex = np.transpose(tex, (1, 0, 2))
            y0 = row_idx * tile_px
            x0 = col_idx * tile_px
            image[y0:y0 + tile_px, x0:x0 + tile_px] = tex.astype(np.uint8)

            # Overlay object sprite if applicable
            if ch in OBJECT_CODES:
                obj_tex = textures.get(OBJECT_CODES[ch], unit)
                obj_tex = np.transpose(obj_tex, (1, 0, 2))
                if obj_tex.shape[-1] == 4:
                    alpha = obj_tex[..., 3:].astype(np.float32) / 255
                    rgb = obj_tex[..., :3].astype(np.float32)
                    base = image[y0:y0 + tile_px, x0:x0 + tile_px].astype(np.float32)
                    blended = alpha * rgb + (1 - alpha) * base
                    image[y0:y0 + tile_px, x0:x0 + tile_px] = blended.astype(np.uint8)
                else:
                    image[y0:y0 + tile_px, x0:x0 + tile_px] = obj_tex[..., :3].astype(np.uint8)

    return image

In [ ]:
grid, area = parse_world_file(WORLD_PATH)
print(f"World: {os.path.basename(WORLD_PATH)}  —  {area[0]}x{area[1]} tiles")

image = render_world(grid, area, tile_px=TILE_PX)

fig, ax = plt.subplots(1, 1, figsize=(8, 8))
ax.imshow(image)
ax.set_title(f"{os.path.basename(WORLD_PATH)} ({area[0]}x{area[1]})")
ax.set_xticks(np.arange(0, area[0] * TILE_PX, TILE_PX) + TILE_PX / 2,
              labels=range(area[0]), fontsize=6)
ax.set_yticks(np.arange(0, area[1] * TILE_PX, TILE_PX) + TILE_PX / 2,
              labels=range(area[1]), fontsize=6)
ax.grid(True, alpha=0.3, linewidth=0.5)
plt.tight_layout()
plt.show()

In [ ]:
# Also print the text grid for reference
print("Tile legend:")
for code, name in sorted(MATERIAL_CODES.items()):
    print(f"  {code} = {name}")
for code, name in sorted(OBJECT_CODES.items()):
    print(f"  {code} = {name} (on grass)")
print()
print("Grid:")
for row in grid:
    print(' '.join(row))